# Piloto / Control — NBC Empresas
Lee la cartera desde Athena, aplica el diseño **solo bajar** (P1 y P2 bajan un nivel; P3/P4 sin intervención) y guarda la asignación.

Diseño: P1 → 20% baja a gestión P2 · P2 → 20% baja a gestión P3 · P3/P4 gestión normal.

In [ ]:
# 1) Imports y configuracion
import awswrangler as wr
import pandas as pd, numpy as np
from scipy import stats

# --- Config de columnas (ajusta si cambian los nombres en la tabla) ---
RUC_COL   = 'numeroruc'
GROUP_COL = 'grupo_propension_cartera'
SCORE_COL = 'score_propension'
STRATA_EXTRA = ['canal']            # se omite solo si no existe en la tabla

MAP_PRIORIDAD = {'1. MUY ALTO':'P1','2. ALTO':'P2','3. MEDIO':'P3','4. BAJO':'P4'}
BAJAR = {'P1':'P2', 'P2':'P3'}      # bandas que se intervienen (agrega 'P3':'P4' si quieres)
FRAC_CONTROL = 0.20
SEED = 2026
SENTINEL = -9.999999999e9

In [ ]:
# 2) Leer la cartera desde Athena como DataFrame
query = """
    SELECT *
    FROM disc_comercial.kbr_base_cartera_despliegue_total
    WHERE periodo_campania = '202606'
"""

df = wr.athena.read_sql_query(query, database='disc_comercial')
print('Filas leidas:', len(df), '| Columnas:', df.shape[1])
df.head()

In [ ]:
# 3) Funciones del piloto (operan sobre el DataFrame en memoria)
def estratos(d): return [c for c in STRATA_EXTRA if c in d.columns]

def preparar(d):
    d = d.copy()
    d[RUC_COL] = d[RUC_COL].astype(str)
    for c in estratos(d):
        d[c] = d[c].astype(str).str.upper().str.strip()
    # dedup por RUC: conserva la prioridad mas alta
    d['_o'] = d[GROUP_COL].astype(str).str.extract(r'^(\d+)').astype(float)
    d = (d.sort_values(['_o', SCORE_COL], ascending=[True, False])
           .drop_duplicates(RUC_COL, keep='first').drop(columns='_o'))
    d['prioridad'] = d[GROUP_COL].map(MAP_PRIORIDAD)
    d = d[d.prioridad.notna()].copy()
    return d.reset_index(drop=True)

def asignar_baja(sub, gestion_baja, frac, rng):
    sub = sub.copy()
    try:
        sub['_q'] = pd.qcut(sub[SCORE_COL], 5, labels=False, duplicates='drop').fillna(-1).astype(int)
    except ValueError:
        sub['_q'] = 0
    keys = estratos(sub) + ['_q']; by = keys if len(keys) > 1 else keys[0]
    sub['rol_experimento'] = 'PILOTO'
    sub['grupo_gestion_asignado'] = sub['prioridad']
    for _, idx in sub.groupby(by).groups.items():
        idx = np.array(idx); rng.shuffle(idx)
        k = max(1, int(round(len(idx)*frac))) if len(idx) >= 2 else 0
        sel = idx[:k]
        sub.loc[sel, 'rol_experimento'] = 'CONTROL'
        sub.loc[sel, 'grupo_gestion_asignado'] = gestion_baja
    return sub.drop(columns='_q')

def ejecutar_piloto(d, frac=FRAC_CONTROL, seed=SEED):
    rng = np.random.default_rng(seed)
    d = preparar(d)
    partes = []
    for banda, baja_a in BAJAR.items():
        sub = d[d.prioridad == banda]
        if len(sub) == 0: continue
        s = asignar_baja(sub, baja_a, frac, rng)
        s['experimento'] = f'EXP_{banda}_baja'
        partes.append(s)
    resto = d[~d.prioridad.isin(BAJAR.keys())].copy()
    if len(resto):
        resto['rol_experimento'] = 'SIN_INTERVENCION'
        resto['grupo_gestion_asignado'] = resto['prioridad']
        resto['experimento'] = 'SIN_INTERVENCION'
        partes.append(resto)
    return pd.concat(partes, ignore_index=True)

In [ ]:
# 4) Aplicar el piloto sobre el DataFrame de Athena
df_asig = ejecutar_piloto(df)

resumen = (df_asig.groupby(['experimento','rol_experimento','grupo_gestion_asignado'])
                  .size().rename('n_rucs').reset_index())
print('Total asignados:', len(df_asig))
resumen

In [ ]:
# 5a) Guardar la asignacion como CSV local
cols = [RUC_COL] + estratos(df_asig) + [GROUP_COL,'prioridad','experimento',
        'rol_experimento','grupo_gestion_asignado',SCORE_COL]
df_asig[cols].to_csv('piloto_control_asignacion_202606.csv', index=False, encoding='utf-8-sig')
print('Guardado: piloto_control_asignacion_202606.csv')

In [ ]:
# 5b) (Opcional) Guardar en S3 y registrar como tabla en Athena
# Descomenta y ajusta el path/base/tabla segun tu entorno.
# wr.s3.to_parquet(
#     df = df_asig[cols],
#     path = 's3://TU-BUCKET/disc_comercial/piloto_control_asignacion/',
#     dataset = True, mode = 'overwrite',
#     database = 'disc_comercial',
#     table = 'kbr_piloto_control_asignacion_202606')
# print('Escrito en S3 + catalogado en Athena')